<a href="https://colab.research.google.com/github/AdelineKwakye/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-99','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item from an order.



### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
lines = pd.json_normalize(
    orders_json,
    record_path = 'lines',
    meta=['order', 'vendor_id']
)
print(lines)
# TODO: assert the row count and the quantity total
assert len(lines) == 4
assert lines['qty'].sum() == 7

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-99


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [10]:
# TODO
vendors = pd.json_normalize(vendors_json) # normalize vendors
joined = lines.merge(
  vendors,
  on = 'vendor_id',
  how = 'left',
  indicator=True
)
print("merged list", joined)

print(joined['_merge'].value_counts())


unmatched = joined[joined['_merge'] == 'left_only']


print("\n Unmatched vendor is: \n", unmatched['vendor_id'].iloc[0])
print("Amount of units attached to the vendor is ", unmatched['qty'].iloc[0])

merged list            item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-99           NaN  NaN  left_only
_merge
both          3
left_only     1
right_only    0
Name: count, dtype: int64

 Unmatched vendor is: 
 V-99
Amount of units attached to the vendor is  3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [11]:
# TODO
joined['name'] = joined['name'].fillna('Unknown')
# changed the unmatched vendor to appear as Unknown because the row is still
# valid so to keep the data I just changed the name to Unknown that way it's not a null value

# group by vendor name
total_quantity = joined.groupby('name')['qty'].sum()

print("total quantiy per vendor name: \n")
print(total_quantity)

total quantiy per vendor name: 

name
Cav Merch       1
Hoos Burgers    3
Unknown         3
Name: qty, dtype: int64


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO
zone_totals = joined.groupby('zone')['qty'].sum()

print(zone_totals)
print("zone total: ", zone_totals.sum())

# the zone total is 4 with zone A have the most items sold.
# the difference is coming from the fact that the unmatched vendor
# has no zone meaning its zone is null so the null zone isn't included, so that's where the missing values went.

zone
A    4
Name: qty, dtype: int64
zone total:  4


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
orders_with_lines = [order for order in ragged_json if 'lines' in order]

print(orders_with_lines) # the dictionaries that include lines

ragged_flat = pd.json_normalize(
    orders_with_lines,
    record_path='lines',
    meta=['order', 'vendor_id']
)

print("\nflatted ragged_json\n")
print(ragged_flat)


[{'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]}, {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]}]

flatted ragged_json

    item  qty order vendor_id
0   Soda  2.0     4      V-01
1  Fries  NaN     6      V-01


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) "One row per line item" is useful because each item and its quantity gets represented as its own row. This makes it easier for someone to analyze the individual items. Evaluating the quantity sold per item becomes a lot easier to read compared to if it was in its orginal nested form, which would make it harder to read and calculate.


b) A quantity of zero means we know that no units were sold, compared to a missing quantity which indicates that the quantity was not given or provided. If we treat missing quantities as zero, downstream calculations such as totals or averages could be incorrect because we would be treating unknown data as if we knew that the quantity was actually zero.